# 약관비교(검증) 로직 테스트 노트북

운영 코드(`app/services/terms_verification_service.py`)와 **동일한 프롬프트/스키마 로직**을 그대로 옮겨와서,
FastAPI 앱·Azure Search·콜백 같은 전체 인프라를 안 띄우고 이 노트북 안에서만 독립적으로 테스트합니다.

## 사용 전 준비
1. 바로 아래 "설정값" 셀에 Azure OpenAI 자격증명을 직접 입력하세요.
2. 이미 OCR 처리된 약관 문서의 결과 텍스트 파일 경로(`parse-di`의 `result.md`, 또는 `document_intelligence_test.ipynb`의 `content.md`)를 입력하세요.
3. "검증 대상 데이터" 셀에서 상품명/항목(`itemNm`/`value`/`desc`)을 원하는 대로 수정해서 테스트하면 됩니다.

> 운영 코드(`terms_verification_service.py`)의 프롬프트/스키마가 나중에 바뀌면, 이 노트북의 해당 셀도 같이 맞춰줘야 합니다.


In [ ]:
# ── 설정값 (직접 입력) ──────────────────────────────────────────
AZURE_OPENAI_ENDPOINT = "https://<your-resource-name>.openai.azure.com/"
AZURE_OPENAI_API_KEY = "<your-api-key>"
AZURE_OPENAI_DEPLOYMENT_NAME = "<deployment-name>"
AZURE_OPENAI_API_VERSION = "2024-12-01-preview"

# 이미 OCR 처리된 약관 문서의 결과 텍스트 파일 경로
# (parse-di의 result.md, 또는 document_intelligence_test.ipynb의 content.md 등)
OCR_RESULT_PATH = r"C:\path\to\result.md"
# ─────────────────────────────────────────────────────────────


## 1. Azure OpenAI 클라이언트 준비 및 약관 문서 로드

In [ ]:
from openai import AzureOpenAI

client = AzureOpenAI(
    azure_endpoint=AZURE_OPENAI_ENDPOINT,
    api_key=AZURE_OPENAI_API_KEY,
    api_version=AZURE_OPENAI_API_VERSION,
)

with open(OCR_RESULT_PATH, encoding="utf-8") as f:
    document_text = f.read()

print(f"문서 길이: {len(document_text)}자")
print(document_text[:300])


## 2. 검증 대상 데이터

실제 `/api/v1/terms/verify` 요청의 `data[].items[]`와 같은 형태입니다. `value`가 `None`이면 status는 항상 `MISMATCH`로
판정되지만, llmValue에는 상품명+항목명을 기준으로 약관에서 찾은 값이 채워집니다.
`value`가 줄바꿈 포함된 여러 조건을 담고 있으면 자동으로 분해(split)되어 각각 검증됩니다.


In [ ]:
# name: 상품명, items: [{itemNm, value, desc}] — 실제 요청 스키마와 동일한 형태
test_data = {
    "name": "요고 69",
    "items": [
        {"itemNm": "초이스 상품여부", "value": "Y", "desc": "선택형 상품 여부"},
        {
            "itemNm": "가입조건",
            "value": (
                "• 듀얼심을 지원하는 단말에 한하여 가입 가능\n"
                "  - USIM 또는 eSIM이 분리되어 메인회선과 듀얼번호를 이용하는 회선이 서로 다른 단말에 장착된 경우 "
                "본 요금제를 이용하는 회선은 일시정지 되며180일간 일시정지 상태가 지속된 경우 해당회선은 해지됨\n"
                "  - 일시정지된 기간은 이용요금을 과금하지 않음\n"
                "• 본 요금제 가입 회선은 약정혜택(단말지원금, 선택약정할인) 및 모든 요금할인을 받을 수 없음\n"
                "• 메인회선과 동일명의인 경우에 한하여 가입 가능하며, 메인회선이 5G 요금제 이용중인 경우에만 가입 가능\n"
                "  - 단, 데이터 전용 요금제, 선불 요금제, 알 기반 청소년 요금제 가입자는 가입 불가\n"
                "• 음성 및 메시지 이용은 메인회선 요금제의 월 제공량을 공유하여 이용\n"
                "  - 메인회선 요금제의 음성/메시지 제공량이 무제한인 경우, 해당 요금제에 적용하는 상업적 이용을 방지하기 위한 "
                "정책을 메인회선과 본 회선에 합산하여 적용"
            ),
            "desc": "듀얼번호 가입조건 전체",
        },
        {"itemNm": "환불 가능 기간", "value": None, "desc": "값 없이 추출 테스트용"},
    ],
}


## 3. 프롬프트/스키마 (운영 코드와 동일)

`app/services/terms_verification_service.py`의 `SYSTEM_PROMPT`, `ITEM_VERIFICATION_SCHEMA`, `CLAIM_SPLIT_SCHEMA`,
`CLAIM_SPLIT_SYSTEM_PROMPT`, `_build_user_prompt`, `_build_split_prompt`를 그대로 옮겨온 것입니다.
프롬프트 문구를 바꿔가며 테스트하고 싶으면 이 셀을 직접 수정하세요.


In [ ]:
SYSTEM_PROMPT = (
    "당신은 약관 문서를 기준으로 상품 항목 데이터를 검증하는 어시스턴트입니다. "
    "반드시 주어진 약관 원문 내용만을 근거로 판단하고, 원문에 없는 내용은 추측하지 마세요.\n"
    "itemNm이나 값이 약관 원문에 완전히 동일한 표현으로 나오지 않아도 괜찮습니다. 동의어·유사 표현·어순 차이 "
    "등으로 표현만 다를 뿐 의미가 같다면 관련 내용/일치하는 것으로 인정하세요. '일치'는 문자 그대로 같은 "
    "표현인지가 아니라 의미가 같은지를 기준으로 판단하세요.\n\n"
    "아래 순서대로 하나씩 판단해서 필드를 채우세요. 뒤 단계는 앞 단계에서 채운 내용을 근거로 판단하세요.\n\n"
    "1. llmValue: 약관 원문 전체(일반 원칙과 예외/단서 조항 모두 포함)를 검토해서 이 항목의 실제 값을 판단해 "
    "채우세요.\n"
    "   - 항목 설명(desc)이 있으면, 항목명(itemNm)이 정확히 무엇을 의미하는지 파악하는 데 참고하세요.\n"
    "   - 현재 값이 없으면, 상품명+항목명을 기준으로 약관에서 값을 찾아 채우세요.\n"
    "   - 약관에 이 항목에 대한 내용 자체가 없으면 null로 하세요.\n"
    "2. evidence: 위에서 판단한 llmValue의 근거가 되는 문장을 약관 원문에서 생략·의역 없이 그대로 인용하세요. "
    "근거가 되는 문장이 여러 곳에 있으면, llmValue를 가장 직접적으로 뒷받침하는 문장 하나만 인용하세요. "
    "llmValue가 null이면 evidence도 null로 하세요.\n"
    "3. page: evidence가 위치한 페이지 번호를 원문의 <!-- PageNumber=\"N\" --> 마커를 참고해 기입하세요. "
    "evidence가 null이거나 페이지를 특정할 수 없으면 null로 하세요.\n"
    "4. article: evidence가 위치한 약관 조항 번호(예: \"제3조\", \"제3조 2항\")가 원문에 표기되어 있으면 "
    "기입하세요. evidence가 null이거나 조항을 특정할 수 없으면 null로 하세요.\n"
    "5. reason: 현재 값과 llmValue가 다른 경우 그 차이를 설명하세요. 현재 값과 llmValue가 완전히 같으면 "
    "null로 하세요.\n"
    "6. status: 1~5에서 채운 내용을 종합해 다음 기준으로 최종 판정하세요. 세 상태는 서로 겹치지 않아야 합니다.\n"
    "   - MATCHED: 현재 값이 llmValue와 완전히 일치 (틀린 내용도 없고 빠진 내용도 없음)\n"
    "   - PARTIAL_MATCH: 현재 값에 llmValue와 다른(틀린) 내용은 없지만, llmValue에 있는 내용 중 일부가 "
    "현재 값에 빠져 있음 (현재 값에 포함된 내용 자체는 모두 맞음)\n"
    "   - MISMATCH: 다음 중 하나 — (a) 현재 값에 llmValue와 다른(틀린) 내용이 하나라도 있음, "
    "(b) 현재 값 자체가 없음, (c) llmValue가 null(약관에 이 항목에 대한 내용 자체가 없음)"
)

ITEM_VERIFICATION_SCHEMA = {
    "name": "terms_item_verification",
    "schema": {
        "type": "object",
        "properties": {
            "llmValue": {"type": ["string", "null"]},
            "evidence": {"type": ["string", "null"]},
            "page": {
                "type": ["integer", "null"],
                "description": "evidence가 위치한 페이지 번호 (원문의 PageNumber 마커 기준)",
            },
            "article": {
                "type": ["string", "null"],
                "description": "evidence가 위치한 약관 조항 번호, 예: '제3조', '제3조 2항'. 특정할 수 없으면 null",
            },
            "reason": {"type": ["string", "null"]},
            "status": {"type": "string", "enum": ["MATCHED", "PARTIAL_MATCH", "MISMATCH"]},
        },
        "required": ["llmValue", "evidence", "page", "article", "reason", "status"],
        "additionalProperties": False,
    },
    "strict": True,
}

CLAIM_SPLIT_SCHEMA = {
    "name": "claim_split",
    "schema": {
        "type": "object",
        "properties": {
            "claims": {"type": "array", "items": {"type": "string"}},
        },
        "required": ["claims"],
        "additionalProperties": False,
    },
    "strict": True,
}

CLAIM_SPLIT_SYSTEM_PROMPT = "당신은 약관 조건 텍스트를 독립적으로 검증 가능한 단위로 분해하는 어시스턴트입니다."


def build_user_prompt(document_text, name, item_nm, desc, claim):
    value_section = f'현재 값: "{claim}"' if claim else "현재 값: (없음, 약관에서 추출 필요)"
    return (
        f"[약관 원문]\n{document_text}\n\n"
        f"[검증 대상]\n"
        f"상품명: {name}\n"
        f"항목명: {item_nm}\n"
        f"항목 설명: {desc or '(없음)'}\n"
        f"{value_section}"
    )


def build_split_prompt(item_nm, value):
    return (
        f"[분해 대상]\n항목명: {item_nm}\n값: {value}\n\n"
        "위 값을 독립적으로 참/거짓 판단이 가능한 조건 단위로 나누세요.\n"
        "- 서로 다른 주제/조건이면 별개 항목으로 분리하세요.\n"
        "- 특정 조건의 예외·결과·부연설명은 관련된 상위 조건에 포함시켜 하나로 유지하세요.\n"
        "- 이미 하나의 독립적인 조건이면 그대로 1개만 반환하세요.\n"
        "- 원본 값에 있는 모든 내용을 빠짐없이 어느 조건 하나에는 포함시키세요. 어떤 내용도 누락하거나 생략하지 마세요."
    )


## 4. 호출 함수 (분해 + 검증)

운영 코드의 `azure_openai_service.create_structured_completion` / `_split_claims` / `_verify_one`과 동일한 로직입니다.
호출마다 토큰 수(캐시 적중 포함)와 소요시간을 같이 출력합니다.


In [ ]:
import json
import time


def call_structured(system_prompt, user_prompt, json_schema):
    start = time.monotonic()
    response = client.chat.completions.create(
        model=AZURE_OPENAI_DEPLOYMENT_NAME,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={"type": "json_schema", "json_schema": json_schema},
    )
    elapsed_ms = (time.monotonic() - start) * 1000
    usage = response.usage
    cached_tokens = 0
    if usage is not None and usage.prompt_tokens_details is not None:
        cached_tokens = usage.prompt_tokens_details.cached_tokens or 0

    return {
        "parsed": json.loads(response.choices[0].message.content),
        "model": response.model,
        "prompt_tokens": usage.prompt_tokens if usage else 0,
        "completion_tokens": usage.completion_tokens if usage else 0,
        "cached_tokens": cached_tokens,
        "elapsed_ms": elapsed_ms,
    }


def split_claims(item_nm, value):
    """value가 없으면 분해 대상이 아니므로 그대로 반환한다."""
    if value is None:
        return [None]
    result = call_structured(CLAIM_SPLIT_SYSTEM_PROMPT, build_split_prompt(item_nm, value), CLAIM_SPLIT_SCHEMA)
    claims = result["parsed"]["claims"]
    print(
        f"  [분해] {item_nm}: {len(claims)}개 조건, "
        f"{result['prompt_tokens']}+{result['completion_tokens']}토큰, {result['elapsed_ms']:.0f}ms"
    )
    return claims


def verify_claim(name, item_nm, desc, original_value, claim):
    result = call_structured(
        SYSTEM_PROMPT, build_user_prompt(document_text, name, item_nm, desc, claim), ITEM_VERIFICATION_SCHEMA
    )
    parsed = result["parsed"]
    print(
        f"  [검증] {item_nm} (claim={claim!r}) -> {parsed['status']}, "
        f"{result['prompt_tokens']}+{result['completion_tokens']}토큰(캐시 {result['cached_tokens']}), {result['elapsed_ms']:.0f}ms"
    )
    return {
        "itemNm": item_nm,
        "value": original_value,
        "subClaim": claim if claim != original_value else None,
        "status": parsed["status"],
        "llmValue": parsed.get("llmValue"),
        "evidence": parsed.get("evidence"),
        "page": parsed.get("page"),
        "article": parsed.get("article"),
        "reason": parsed.get("reason"),
    }


## 5. 전체 실행 (분해 → 검증)

In [ ]:
all_results = []

for item in test_data["items"]:
    item_nm = item["itemNm"]
    value = item.get("value")
    desc = item.get("desc")
    print(f"\n=== {item_nm} (value={value!r}) ===")

    claims = split_claims(item_nm, value)
    for claim in claims:
        result = verify_claim(test_data["name"], item_nm, desc, value, claim)
        all_results.append(result)

print("\n\n=== 최종 결과 ===")
print(json.dumps(all_results, ensure_ascii=False, indent=2))
